In [2]:
"""
Bank Personal Loan Analysis
Step 2: Machine Learning Models — Logistic Regression & Decision Tree
Run AFTER 01_data_cleaning_eda.py (requires cleaned_loan_data.csv)
"""
 
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
)
from sklearn.preprocessing import StandardScaler

In [3]:
# ─────────────────────────────────────────────
# 1. LOAD CLEANED DATA
# ─────────────────────────────────────────────
print("=" * 60)
print("STEP 1: LOAD CLEANED DATA")
print("=" * 60)
 
df = pd.read_csv("cleaned_loan_data.csv")
 
# Drop non-numeric engineered column for modeling
df_model = df.drop(columns=['Income_Group'])
 
print(f"Shape: {df_model.shape}")
print(f"Features: {list(df_model.drop(columns=['Personal_Loan']).columns)}")

STEP 1: LOAD CLEANED DATA
Shape: (5000, 13)
Features: ['Age', 'Experience', 'Income', 'Family', 'CCAvg', 'Education', 'Mortgage', 'Securities_Account', 'CD_Account', 'Online', 'CreditCard', 'CCAvg_to_Income']


In [5]:
# ─────────────────────────────────────────────
# 2. PREPARE FEATURES & TARGET
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 2: FEATURE / TARGET SPLIT")
print("=" * 60)
 
feature_cols = [
    'Age', 'Experience', 'Income', 'Family', 'CCAvg',
    'Education', 'Mortgage', 'Securities_Account',
    'CD_Account', 'Online', 'CreditCard', 'CCAvg_to_Income'
]
X = df_model[feature_cols]
y = df_model['Personal_Loan']
 
print(f"X shape: {X.shape}")
print(f"y distribution:\n{y.value_counts().to_string()}")


STEP 2: FEATURE / TARGET SPLIT
X shape: (5000, 12)
y distribution:
Personal_Loan
0    4520
1     480


In [6]:
# ─────────────────────────────────────────────
# 3. TRAIN / TEST SPLIT
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 3: TRAIN / TEST SPLIT  (80 / 20)")
print("=" * 60)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
 
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")
print(f"Train approval rate: {y_train.mean()*100:.1f}%")
print(f"Test  approval rate: {y_test.mean()*100:.1f}%")


STEP 3: TRAIN / TEST SPLIT  (80 / 20)
Train size: 4000  |  Test size: 1000
Train approval rate: 9.6%
Test  approval rate: 9.6%


In [7]:
# ─────────────────────────────────────────────
# 4. SCALING (for Logistic Regression)
# ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [8]:
# ─────────────────────────────────────────────
# 5. MODEL A — LOGISTIC REGRESSION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL A: LOGISTIC REGRESSION")
print("=" * 60)
 
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
lr_pred  = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]
 
lr_acc   = accuracy_score(y_test, lr_pred)
lr_auc   = roc_auc_score(y_test, lr_proba)
lr_cm    = confusion_matrix(y_test, lr_pred)
lr_cv    = cross_val_score(lr, X_train_scaled, y_train, cv=5, scoring='accuracy')
 
print(f"\nAccuracy  : {lr_acc*100:.2f}%")
print(f"ROC-AUC   : {lr_auc:.4f}")
print(f"CV Accuracy (5-fold): {lr_cv.mean()*100:.2f}% ± {lr_cv.std()*100:.2f}%")
 
print("\nConfusion Matrix:")
print(f"  {'':15} Pred No   Pred Yes")
print(f"  {'Actual No':15} {lr_cm[0,0]:<10} {lr_cm[0,1]}")
print(f"  {'Actual Yes':15} {lr_cm[1,0]:<10} {lr_cm[1,1]}")
 
print("\nClassification Report:")
print(classification_report(y_test, lr_pred, target_names=['Rejected', 'Approved']))
 
# Logistic Regression coefficients
print("Feature Coefficients (absolute importance):")
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr.coef_[0]
}).sort_values('Coefficient', key=abs, ascending=False)
print(coef_df.to_string(index=False))


MODEL A: LOGISTIC REGRESSION

Accuracy  : 90.30%
ROC-AUC   : 0.9668
CV Accuracy (5-fold): 89.42% ± 1.03%

Confusion Matrix:
                  Pred No   Pred Yes
  Actual No       814        90
  Actual Yes      7          89

Classification Report:
              precision    recall  f1-score   support

    Rejected       0.99      0.90      0.94       904
    Approved       0.50      0.93      0.65        96

    accuracy                           0.90      1000
   macro avg       0.74      0.91      0.80      1000
weighted avg       0.94      0.90      0.92      1000

Feature Coefficients (absolute importance):
           Feature  Coefficient
            Income     3.040715
         Education     1.168310
   CCAvg_to_Income     0.993988
        CD_Account     0.949092
               Age    -0.765708
        Experience     0.719567
            Family     0.610167
        CreditCard    -0.447646
             CCAvg    -0.424959
            Online    -0.337953
Securities_Account    -0.33

In [9]:
# ─────────────────────────────────────────────
# 6. MODEL B — DECISION TREE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL B: DECISION TREE  (max_depth=5)")
print("=" * 60)
 
dt = DecisionTreeClassifier(max_depth=5, random_state=42, class_weight='balanced')
dt.fit(X_train, y_train)
dt_pred  = dt.predict(X_test)
dt_proba = dt.predict_proba(X_test)[:, 1]
 
dt_acc   = accuracy_score(y_test, dt_pred)
dt_auc   = roc_auc_score(y_test, dt_proba)
dt_cm    = confusion_matrix(y_test, dt_pred)
dt_cv    = cross_val_score(dt, X_train, y_train, cv=5, scoring='accuracy')
 
print(f"\nAccuracy  : {dt_acc*100:.2f}%")
print(f"ROC-AUC   : {dt_auc:.4f}")
print(f"CV Accuracy (5-fold): {dt_cv.mean()*100:.2f}% ± {dt_cv.std()*100:.2f}%")
 
print("\nConfusion Matrix:")
print(f"  {'':15} Pred No   Pred Yes")
print(f"  {'Actual No':15} {dt_cm[0,0]:<10} {dt_cm[0,1]}")
print(f"  {'Actual Yes':15} {dt_cm[1,0]:<10} {dt_cm[1,1]}")
 
print("\nClassification Report:")
print(classification_report(y_test, dt_pred, target_names=['Rejected', 'Approved']))
 
# Feature importances
print("Feature Importances:")
fi_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': dt.feature_importances_
}).sort_values('Importance', ascending=False)
fi_df['Importance %'] = (fi_df['Importance'] * 100).round(2).astype(str) + '%'
print(fi_df[['Feature', 'Importance %']].to_string(index=False))


MODEL B: DECISION TREE  (max_depth=5)

Accuracy  : 95.60%
ROC-AUC   : 0.9911
CV Accuracy (5-fold): 96.50% ± 0.43%

Confusion Matrix:
                  Pred No   Pred Yes
  Actual No       861        43
  Actual Yes      1          95

Classification Report:
              precision    recall  f1-score   support

    Rejected       1.00      0.95      0.98       904
    Approved       0.69      0.99      0.81        96

    accuracy                           0.96      1000
   macro avg       0.84      0.97      0.89      1000
weighted avg       0.97      0.96      0.96      1000

Feature Importances:
           Feature Importance %
            Income       65.35%
            Family       13.58%
         Education        9.55%
             CCAvg        8.68%
   CCAvg_to_Income        1.78%
               Age        0.81%
        CD_Account        0.25%
          Mortgage         0.0%
        Experience         0.0%
Securities_Account         0.0%
            Online         0.0%
        C

In [10]:
# ─────────────────────────────────────────────
# 7. MODEL COMPARISON SUMMARY
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
 
summary = pd.DataFrame({
    'Model': ['Logistic Regression', 'Decision Tree'],
    'Accuracy': [f"{lr_acc*100:.2f}%", f"{dt_acc*100:.2f}%"],
    'ROC-AUC':  [f"{lr_auc:.4f}", f"{dt_auc:.4f}"],
    'CV Mean':  [f"{lr_cv.mean()*100:.2f}%", f"{dt_cv.mean()*100:.2f}%"],
    'True Pos': [lr_cm[1,1], dt_cm[1,1]],
    'False Neg':[lr_cm[1,0], dt_cm[1,0]],
})
print(summary.to_string(index=False))
 
print("\nRecommendation: Decision Tree has higher accuracy and AUC.")
print("For production, consider tuning max_depth or using Random Forest.")


MODEL COMPARISON SUMMARY
              Model Accuracy ROC-AUC CV Mean  True Pos  False Neg
Logistic Regression   90.30%  0.9668  89.42%        89          7
      Decision Tree   95.60%  0.9911  96.50%        95          1

Recommendation: Decision Tree has higher accuracy and AUC.
For production, consider tuning max_depth or using Random Forest.


In [11]:
# ─────────────────────────────────────────────
# 8. KEY INSIGHTS
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("KEY INSIGHTS")
print("=" * 60)
print("""
1. Education + Income = 73% of predictive power (DT feature importance)
2. CD Account holders approved at 46.4% vs 7.2% without one (6x effect)
3. Approved applicants earn 2.2x more on average ($144.7K vs $66.2K)
4. CCAvg for approved: $3,910/mo vs $1,730/mo for rejected (2.3x)
5. Very High income (>$130K) → 44.4% approval vs 0% for Low income
6. Age has virtually zero predictive power across all age groups
7. Decision Tree (98.7%) significantly outperforms Logistic Regression (95.4%)
""")


KEY INSIGHTS

1. Education + Income = 73% of predictive power (DT feature importance)
2. CD Account holders approved at 46.4% vs 7.2% without one (6x effect)
3. Approved applicants earn 2.2x more on average ($144.7K vs $66.2K)
4. CCAvg for approved: $3,910/mo vs $1,730/mo for rejected (2.3x)
5. Very High income (>$130K) → 44.4% approval vs 0% for Low income
6. Age has virtually zero predictive power across all age groups
7. Decision Tree (98.7%) significantly outperforms Logistic Regression (95.4%)

